## Mamba Vision 权重裁剪

In [6]:
from safetensors.torch import load_file, save_file

# 加载 safetensors 权重字典
pretrained_dict = load_file("model.safetensors")

In [5]:
print(f"Total keys: {len(pretrained_dict.keys())}")

# 打印前 20 个 Key 和它们的形状，看看它是怎么命名的
for i, (key, tensor) in enumerate(pretrained_dict.items()):
    print(f"{key}: {tensor.shape}")
    if i >= 20: 
        break

Total keys: 356
model.levels.0.blocks.0.norm1.num_batches_tracked: torch.Size([])
model.levels.0.blocks.0.norm2.num_batches_tracked: torch.Size([])
model.levels.0.blocks.1.norm1.num_batches_tracked: torch.Size([])
model.levels.0.blocks.1.norm2.num_batches_tracked: torch.Size([])
model.levels.0.blocks.2.norm1.num_batches_tracked: torch.Size([])
model.levels.0.blocks.2.norm2.num_batches_tracked: torch.Size([])
model.levels.1.blocks.0.norm1.num_batches_tracked: torch.Size([])
model.levels.1.blocks.0.norm2.num_batches_tracked: torch.Size([])
model.levels.1.blocks.1.norm1.num_batches_tracked: torch.Size([])
model.levels.1.blocks.1.norm2.num_batches_tracked: torch.Size([])
model.levels.1.blocks.2.norm1.num_batches_tracked: torch.Size([])
model.levels.1.blocks.2.norm2.num_batches_tracked: torch.Size([])
model.norm.num_batches_tracked: torch.Size([])
model.patch_embed.conv_down.1.num_batches_tracked: torch.Size([])
model.patch_embed.conv_down.4.num_batches_tracked: torch.Size([])
model.head.bi

In [8]:
backbone_weights = {}

for key, tensor in pretrained_dict.items():
    # 步骤 A：遇到 'head' 直接跳过，坚决不要
    if "head" in key:
        continue
        
    # 步骤 B：把开头的 'model.' 前缀去掉，方便嫁接
    # 例如把 "model.levels.0..." 变成 "levels.0..."
    if key.startswith("model."):
        new_key = key.replace("model.", "", 1)
    else:
        new_key = key
        
    # 把过滤和改名后的权重存入新字典
    backbone_weights[new_key] = tensor

print(f"原始包含 Head 的权重数: {len(pretrained_dict)}")
print(f"提取后纯 Backbone 的权重数: {len(backbone_weights)}")

# 3. 保存为你专属的 Backbone 权重文件
save_file(backbone_weights, "mambavision_backbone_clean.safetensors")
print("搞定！干净的 Backbone 权重已保存。")

原始包含 Head 的权重数: 356
提取后纯 Backbone 的权重数: 354
搞定！干净的 Backbone 权重已保存。


## RT-DETR 的 SSM+Heads裁剪

In [9]:
import torch

# 1. 加载下载好的 RT-DETR R101 权重
checkpoint_path = "./rtdetr_r101vd_6x_coco_from_paddle.pth"
checkpoint = torch.load(checkpoint_path, map_location="cpu")

# 提取真正的模型权重字典 (具体取决于官方保存的 key 叫 'model' 还是 'state_dict' 还是 'ema')
if 'ema' in checkpoint and checkpoint['ema'] is not None:
    original_weights = checkpoint['ema']['module'] # RT-DETR 官方库习惯存 ema
elif 'model' in checkpoint:
    original_weights = checkpoint['model']
else:
    original_weights = checkpoint

# 2. 准备一个新的字典，只装 Neck 和 Head
neck_head_weights = {}

for key, tensor in original_weights.items():
    # 过滤掉 Backbone：ResNet 的权重通常以 'backbone.' 开头
    # 有的实现中可能是 'backbone.res...' 或者类似名字，直接粗暴剔除 'backbone'
    if key.startswith("backbone."):
        continue
    
    # 保留剩下的部分 (Encoder/AIFI, CCFM, Decoder/Head)
    neck_head_weights[key] = tensor

print(f"原始权重总数: {len(original_weights)}")
print(f"剔除 Backbone 后剩余权重数: {len(neck_head_weights)}")

# 3. 保存为新的干净权重 (推荐存为 safetensors，或者直接存 pth)
# torch.save(neck_head_weights, "rtdetr_r101_neck_head_only.pth")
from safetensors.torch import save_file
save_file(neck_head_weights, "rtdetr_r101_neck_head_only.safetensors")

print("RT-DETR 的 Neck+Head 提取成功！")

原始权重总数: 991
剔除 Backbone 后剩余权重数: 461
RT-DETR 的 Neck+Head 提取成功！
